# 공유 스모크 테스트 패키지 재실행

팀원이 공유자의 이미지 4장·라벨로 같은 검사를 수행하는 Notebook입니다.
전체 학습 데이터는 필요하지 않습니다.

1. 같은 브랜치의 코드를 받습니다. `detector/`, `detector_data/`, `detector_data/detector_data_smoke_test/`가 있어야 합니다.
2. Google Drive에서 `smoke_test.zip`을 받고 저장소의 `datasets/smoke_test/`에 압축을 풉니다.
3. 저장소의 학습 환경을 커널로 선택하고 이 Notebook을 저장소 안에서 실행합니다.
4. 아래 셀을 위에서부터 실행합니다.

검사는 한 배치로 종료되며 50 epoch 학습이나 checkpoint 저장은 하지 않습니다.


## 1. 실행 위치와 Python 확인


In [ ]:
from pathlib import Path
import subprocess
import sys

# 현재 위치 또는 상위 폴더에서 공유 저장소 찾기
project_root = next(
    (
        path
        for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "detector").is_dir()
        and (path / "detector_data").is_dir()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        "공유 저장소 안에 Notebook을 저장한 뒤 실행하세요."
    )

print("실행 기준:", project_root)
print("Python:", sys.executable)


## 2. 같은 패키지·코드인지 확인

이미지·라벨과 코드 해시를 확인합니다. 공유자가 알려준 패키지 식별값과 비교하세요.
이 셀은 모델을 실행하지 않습니다.


In [ ]:
subprocess.run(
    [
        sys.executable, "-m",
        "detector_data.detector_data_smoke_test.smoke_bundle",
        "check",
    ],
    cwd=project_root,
    check=True,
)

## 3. 공유 표본으로 모델 연결 검사

Train 2장·Validation 2장을 읽고 입력 형식, 손실 계산·역전파·optimizer 1회,
Validation 예측을 확인합니다. 기본 장치는 CPU입니다.

통과 기준은 `모델 연결 검사: passed`와 `[통과] 공유 스모크 패키지 검사 완료`입니다.
장치·라이브러리가 다르면 손실값까지 같을 필요는 없습니다.


In [ ]:
subprocess.run(
    [
        sys.executable, "-m",
        "detector_data.detector_data_smoke_test.smoke_bundle",
        "run", "--model-step",
    ],
    cwd=project_root,
    check=True,
)

## 결과 기록

검사 결과는 outputs/detector_data/shared_smoke_report.json에 저장됩니다. 이 파일은 Git에 올리지 않습니다.